# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdityaAAND/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pyspark.sql import SparkSession

# Create a local Spark session using all available cores
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_Spark_Test") \
    .getOrCreate()

# Verify the session
print(spark)



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I split each client's timeline at its own midpoint between gsc_data_start and snapshot end.Features come from ebfore the cutoff and labels from after.This is true for an early warning question because here I'm predicting the future of the same pairs,not generalizing to new clients, so a client-holdout split would test wrong.And here I'm using per-client cutoff instead of one global date as clients have very different history lengths,one shared date would give long-history clients a lopsided split and short-history clients with barely any data on one side

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from huggingface_hub import HfApi
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
api = HfApi(token=hf_token)



from datasets import load_dataset

ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance",streaming = True, split="train")
df = ds.to_pandas()
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
con.sql("SELECT COUNT(*) FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
con.execute("PRAGMA memory_limit='3GB'")   # match your Colab RAM tier, leave headroom
con.execute("PRAGMA temp_directory='/content/duckdb_tmp'")
rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# Split design check — time-aware, per-client cutoff (not one global date)

fact_rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"
clients_rel = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

# 1. Get each client's own history start, from dim_clients (only clients with real GSC access)
clients = con.sql(f"""
    SELECT client_hash_id, gsc_data_start
    FROM read_parquet('{clients_rel}')
    WHERE has_gsc_access IS TRUE
    AND gsc_data_start IS NOT NULL
""").df()

# 2. Snapshot end date
snapshot_end = con.sql(f"""
    SELECT MAX(report_date) AS max_date
    FROM read_parquet('{fact_rel}')
""").df().loc[0, "max_date"]
print(f"Snapshot end date: {snapshot_end}")

# 3. Per-client cutoff = midpoint of (gsc_data_start, snapshot_end)
clients["cutoff_date"] = clients["gsc_data_start"] + (snapshot_end - clients["gsc_data_start"]) / 2
print(clients[["client_hash_id", "gsc_data_start", "cutoff_date"]].head())

con.register("client_cutoffs", clients[["client_hash_id", "cutoff_date"]])

# 4. Row counts on each side of the PER-CLIENT cutoff — only rows with real GSC data
counts = con.sql(f"""
    SELECT
        SUM(CASE WHEN f.report_date < c.cutoff_date THEN 1 ELSE 0 END) AS feature_rows,
        SUM(CASE WHEN f.report_date >= c.cutoff_date THEN 1 ELSE 0 END) AS label_rows
    FROM read_parquet('{fact_rel}') f
    JOIN client_cutoffs c ON f.client_hash_id = c.client_hash_id
    WHERE f.gsc_data_available IS TRUE
""").df()
print(counts)

# 5. Sanity check: every content item has real GSC data on both sides of ITS client's cutoff
pairs_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_content_items,
        SUM(CASE WHEN has_before AND has_after THEN 1 ELSE 0 END) AS items_with_both
    FROM (
        SELECT
            f.content_hash_id,
            BOOL_OR(f.report_date < c.cutoff_date) AS has_before,
            BOOL_OR(f.report_date >= c.cutoff_date) AS has_after
        FROM read_parquet('{fact_rel}') f
        JOIN client_cutoffs c ON f.client_hash_id = c.client_hash_id
        WHERE f.gsc_data_available IS TRUE
        GROUP BY f.content_hash_id
    )
""").df()
print(pairs_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Snapshot end date: 2026-06-30 00:00:00
            client_hash_id gsc_data_start         cutoff_date
0  client_06d356715a8ff3b6     2026-04-10 2026-05-20 12:00:00
1  client_08a6a72ff48e62c0     2025-09-24 2026-02-10 12:00:00
2  client_08d2847f24cf89c1     2025-07-21 2026-01-09 00:00:00
3  client_0b245132bb722950     2026-04-12 2026-05-21 12:00:00
4  client_0e1acc6cd57b0eba     2025-09-24 2026-02-10 12:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   feature_rows  label_rows
0     8950079.0  19504192.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_content_items  items_with_both
0               297494         167964.0


In [ ]:
query_rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet"
size_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM read_parquet('{query_rel}')
""").df()
print(size_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  n_clients
0     2414248         52


In [ ]:
size_at_various_caps = con.sql(f"""
    WITH pop AS (
        SELECT client_hash_id, query_hash_id, COUNT(*) AS n_content
        FROM read_parquet('{query_rel}')
        GROUP BY client_hash_id, query_hash_id
    )
    SELECT
        SUM(CASE WHEN n_content BETWEEN 2 AND 10 THEN n_content*(n_content-1)/2 ELSE 0 END) AS pairs_if_cap_10,
        SUM(CASE WHEN n_content BETWEEN 2 AND 15 THEN n_content*(n_content-1)/2 ELSE 0 END) AS pairs_if_cap_15,
        SUM(CASE WHEN n_content BETWEEN 2 AND 25 THEN n_content*(n_content-1)/2 ELSE 0 END) AS pairs_if_cap_25,
        SUM(CASE WHEN n_content BETWEEN 2 AND 50 THEN n_content*(n_content-1)/2 ELSE 0 END) AS pairs_if_cap_50
    FROM pop
""").df()
print(size_at_various_caps)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   pairs_if_cap_10  pairs_if_cap_15  pairs_if_cap_25  pairs_if_cap_50
0        2031191.0        2652554.0        3356832.0        4095934.0


In [ ]:
con.execute("PRAGMA threads=2")
con.execute("PRAGMA preserve_insertion_order=false")
con.execute("PRAGMA memory_limit='3GB'")
con.execute("PRAGMA temp_directory='/content/duckdb_tmp'")

# Stage 1: pull content_queries ONCE into a local table
con.execute(f"""
    CREATE OR REPLACE TABLE content_queries AS
    SELECT DISTINCT client_hash_id, content_hash_id, query_hash_id
    FROM read_parquet('{query_rel}')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# Stage 2: query popularity, materialized
con.execute(f"""
    CREATE OR REPLACE TABLE query_popularity AS
    SELECT client_hash_id, query_hash_id, COUNT(*) AS n_content
    FROM content_queries
    GROUP BY client_hash_id, query_hash_id
    HAVING COUNT(*) BETWEEN 2 AND 10
""")

In [ ]:
# Stage 3: filtered content_queries, materialized (this is the one used twice in self-join)
con.execute("""
    CREATE OR REPLACE TABLE filtered_cq AS
    SELECT cq.*
    FROM content_queries cq
    JOIN query_popularity qp
        ON cq.client_hash_id = qp.client_hash_id AND cq.query_hash_id = qp.query_hash_id
""")
print(con.sql("SELECT COUNT(*) FROM filtered_cq").df())

   count_star()
0       1315427


In [ ]:
# Stage 4: query set sizes, materialized
con.execute("""
    CREATE OR REPLACE TABLE query_set_sizes AS
    SELECT client_hash_id, content_hash_id, COUNT(*) AS n_queries
    FROM content_queries
    GROUP BY client_hash_id, content_hash_id
""")

In [ ]:
# Stage 5: now the self-join reads from real tables, not recomputed CTEs
pairs = con.sql("""
    SELECT s.client_hash_id, s.content_a, s.content_b, s.shared_queries,
           sa.n_queries AS n_queries_a, sb.n_queries AS n_queries_b,
           s.shared_queries * 1.0 / (sa.n_queries + sb.n_queries - s.shared_queries) AS jaccard
    FROM (
        SELECT a.client_hash_id, a.content_hash_id AS content_a, b.content_hash_id AS content_b,
               COUNT(*) AS shared_queries
        FROM filtered_cq a
        JOIN filtered_cq b
            ON a.client_hash_id = b.client_hash_id AND a.query_hash_id = b.query_hash_id
            AND a.content_hash_id < b.content_hash_id
        GROUP BY a.client_hash_id, a.content_hash_id, b.content_hash_id
    ) s
    JOIN query_set_sizes sa ON s.client_hash_id = sa.client_hash_id AND s.content_a = sa.content_hash_id
    JOIN query_set_sizes sb ON s.client_hash_id = sb.client_hash_id AND s.content_b = sb.content_hash_id
    WHERE s.shared_queries * 1.0 / (sa.n_queries + sb.n_queries - s.shared_queries) >= 0.2
    ORDER BY jaccard DESC
""").df()

print(f"Candidate pairs found: {len(pairs):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Candidate pairs found: 79,694


In [ ]:
print(con.sql("SHOW TABLES").df())

               name
0    client_cutoffs
1   content_queries
2       filtered_cq
3  query_popularity
4   query_set_sizes


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE pair_content_ids AS
    SELECT DISTINCT content_a AS content_hash_id, client_hash_id FROM pairs
    UNION
    SELECT DISTINCT content_b AS content_hash_id, client_hash_id FROM pairs
""")
print(con.sql("SELECT COUNT(*) FROM pair_content_ids").df())

   count_star()
0         63643


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE content_daily AS
    SELECT f.client_hash_id, f.content_hash_id, f.report_date, f.gsc_impressions
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet') f
    JOIN pair_content_ids pc
        ON f.client_hash_id = pc.client_hash_id AND f.content_hash_id = pc.content_hash_id
    JOIN client_cutoffs c ON f.client_hash_id = c.client_hash_id
    WHERE f.report_date < c.cutoff_date AND f.gsc_data_available IS TRUE
""")
print(con.sql("SELECT COUNT(*) FROM content_daily").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   count_star()
0       3325131


In [ ]:
share_trend = con.sql("""
WITH pair_daily AS (
    SELECT
        p.client_hash_id, p.content_a, p.content_b, da.report_date,
        da.gsc_impressions AS impr_a, db.gsc_impressions AS impr_b
    FROM pairs p
    JOIN content_daily da ON p.client_hash_id = da.client_hash_id AND p.content_a = da.content_hash_id
    JOIN content_daily db ON p.client_hash_id = db.client_hash_id AND p.content_b = db.content_hash_id AND da.report_date = db.report_date
),
pair_share AS (
    SELECT client_hash_id, content_a, content_b, report_date, impr_a, impr_b,
        CASE WHEN (impr_a + impr_b) > 0 THEN impr_a * 1.0 / (impr_a + impr_b) ELSE NULL END AS share_a
    FROM pair_daily
)
SELECT client_hash_id, content_a, content_b, COUNT(*) AS n_days,
    REGR_SLOPE(share_a, EPOCH(report_date)) AS share_a_trend_slope,
    AVG(share_a) AS avg_share_a
FROM pair_share
WHERE share_a IS NOT NULL
GROUP BY client_hash_id, content_a, content_b
HAVING COUNT(*) >= 10
""").df()
print(f"Pairs with share-trend feature: {len(share_trend):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pairs with share-trend feature: 40,036


In [ ]:
print(f"Pairs with share-trend feature: {len(share_trend):,}")
share_trend.head()

Pairs with share-trend feature: 40,036


,client_hash_id,content_a,content_b,n_days,share_a_trend_slope,avg_share_a
0,client_7eafe750768f0ac2,content_84f75b15bec3ef49,content_a839cf96934a0b11,12,-8.370842e-07,0.641540
1,client_a80fca3f171ed1de,content_94336127af47ae07,content_c53905896ae59067,37,-6.528659e-08,0.266040
2,client_a80fca3f171ed1de,content_aa30ebefc7989af1,content_f8001c53dcbbff6b,15,-1.182990e-08,0.730509
3,client_157ffe4d4a595515,content_6cea199b4e488fc9,content_e308267d8656c3a4,44,1.246147e-08,0.419860
4,client_3f0ce4d44fe94f3d,content_8544be8542f4e19a,content_d9557006c14ff929,17,-1.100617e-07,0.749141


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE content_daily_position AS
    SELECT f.client_hash_id, f.content_hash_id, f.report_date, f.gsc_avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet') f
    JOIN pair_content_ids pc
        ON f.client_hash_id = pc.client_hash_id AND f.content_hash_id = pc.content_hash_id
    JOIN client_cutoffs c ON f.client_hash_id = c.client_hash_id
    WHERE f.report_date < c.cutoff_date
      AND f.gsc_data_available IS TRUE
      AND f.gsc_avg_position > 0   -- 0 means "no position data", not position zero
""")
print(con.sql("SELECT COUNT(*) FROM content_daily_position").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   count_star()
0       3210571


In [ ]:
position_corr = con.sql("""
WITH pair_daily_pos AS (
    SELECT
        p.client_hash_id, p.content_a, p.content_b, da.report_date,
        da.gsc_avg_position AS pos_a, db.gsc_avg_position AS pos_b
    FROM pairs p
    JOIN content_daily_position da ON p.client_hash_id = da.client_hash_id AND p.content_a = da.content_hash_id
    JOIN content_daily_position db ON p.client_hash_id = db.client_hash_id AND p.content_b = db.content_hash_id
        AND da.report_date = db.report_date
)
SELECT
    client_hash_id, content_a, content_b,
    COUNT(*) AS n_days,
    CORR(pos_a, pos_b) AS position_correlation
FROM pair_daily_pos
GROUP BY client_hash_id, content_a, content_b
HAVING COUNT(*) >= 10
""").df()

print(f"Pairs with position-correlation feature: {len(position_corr):,}")
position_corr.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pairs with position-correlation feature: 39,227


,client_hash_id,content_a,content_b,n_days,position_correlation
0,client_9958f0a7ae1df715,content_31a778607b7b0116,content_543bdc4fa89a36d3,193,0.271618
1,client_9958f0a7ae1df715,content_3b8c8fe1926c94d9,content_4fdc46e773568767,200,0.511351
2,client_73cda7b4e4f265ea,content_dd8c19687cb28c30,content_f137855d00239020,185,0.050893
3,client_73cda7b4e4f265ea,content_832b8e61abb850fb,content_a9ad4df3d6647a79,246,0.629469
4,client_73cda7b4e4f265ea,content_a61635ecfbb9106f,content_d1a229d7ec0e9fc5,252,0.090287


In [ ]:
# Check dim_content's schema first — haven't confirmed its columns yet
content_rel = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
describe_content = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{content_rel}') LIMIT 1").df()
print(describe_content.to_string())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [ ]:
content_rel = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

freshness_gap = con.sql(f"""
WITH content_meta AS (
    SELECT client_hash_id, content_hash_id, content_updated_date, content_created_date
    FROM read_parquet('{content_rel}')
    WHERE is_deleted IS NOT TRUE
)
SELECT
    p.client_hash_id, p.content_a, p.content_b,
    ma.content_updated_date AS updated_a, mb.content_updated_date AS updated_b,
    ABS(DATE_DIFF('day', ma.content_updated_date, mb.content_updated_date)) AS freshness_gap_days,
    ma.content_created_date AS created_a, mb.content_created_date AS created_b,
    ABS(DATE_DIFF('day', ma.content_created_date, mb.content_created_date)) AS age_gap_days
FROM pairs p
JOIN content_meta ma ON p.client_hash_id = ma.client_hash_id AND p.content_a = ma.content_hash_id
JOIN content_meta mb ON p.client_hash_id = mb.client_hash_id AND p.content_b = mb.content_hash_id
""").df()

print(f"Pairs with freshness-gap feature: {len(freshness_gap):,}")
freshness_gap.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pairs with freshness-gap feature: 79,693


,client_hash_id,content_a,content_b,updated_a,updated_b,freshness_gap_days,created_a,created_b,age_gap_days
0,client_2094c6eb080311d5,content_14c416ddfd2bd2b0,content_f455592a451e8094,2026-06-01,2026-05-20,12,2026-05-02,2026-03-06,57
1,client_2094c6eb080311d5,content_14ef58c38dd0ff6f,content_46dc5dd85ab2c8a3,2026-05-20,2026-05-12,8,2026-02-10,2026-02-04,6
2,client_2094c6eb080311d5,content_14feb5514613e44d,content_240b7751ead8a1fd,2026-06-22,2026-05-20,33,2026-03-24,2026-04-30,37
3,client_2094c6eb080311d5,content_15200126d986d645,content_50b737fe4541b75a,2026-05-20,2026-05-20,0,2026-03-24,2026-03-24,0
4,client_2094c6eb080311d5,content_15565677b6e1792f,content_b8a3530043b2195e,2026-05-20,2026-05-20,0,2026-03-11,2026-03-06,5


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/flyrank_capstone'
os.makedirs(save_dir, exist_ok=True)

Mounted at /content/drive


In [ ]:
tables_to_save = [
    'pairs', 'client_cutoffs', 'content_daily', 'content_daily_position',
    'pair_content_ids', 'content_queries', 'filtered_cq',
    'query_popularity', 'query_set_sizes'
]

for t in tables_to_save:
    try:
        con.sql(f"COPY {t} TO '{save_dir}/{t}.parquet' (FORMAT PARQUET)")
        print(f"Saved {t}")
    except Exception as e:
        print(f"Skipped {t}: {e}")

Saved pairs
Saved client_cutoffs


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved content_daily
Saved content_daily_position
Saved pair_content_ids
Saved content_queries
Saved filtered_cq
Saved query_popularity
Saved query_set_sizes


In [ ]:
share_trend.to_parquet(f'{save_dir}/share_trend.parquet')
position_corr.to_parquet(f'{save_dir}/position_corr.parquet')
freshness_gap.to_parquet(f'{save_dir}/freshness_gap.parquet')

In [1]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/flyrank_capstone'

# 2. Re-auth for HF (only needed if you'll touch read_parquet('hf://...') again)
from getpass import getpass
hf_token = getpass("Paste your Hugging Face read token: ")

# 3. Fresh DuckDB connection + pragmas
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA preserve_insertion_order=false")
con.execute("PRAGMA memory_limit='3GB'")
con.execute("PRAGMA temp_directory='/content/duckdb_tmp'")

# 4. Reload every saved DuckDB table from Drive instead of recomputing
tables_to_save = [
    'pairs', 'client_cutoffs', 'content_daily', 'content_daily_position',
    'pair_content_ids', 'content_queries', 'filtered_cq',
    'query_popularity', 'query_set_sizes'
]
for t in tables_to_save:
    con.execute(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM read_parquet('{save_dir}/{t}.parquet')")
    print(f"Loaded {t}")

# 5. Reload the pandas DataFrame features
import pandas as pd
share_trend = pd.read_parquet(f'{save_dir}/share_trend.parquet')
position_corr = pd.read_parquet(f'{save_dir}/position_corr.parquet')
freshness_gap = pd.read_parquet(f'{save_dir}/freshness_gap.parquet')

# 6. Also re-set query_rel / fact_rel / clients_rel / content_rel — small variables, cheap to redefine
query_rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet"
fact_rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"
clients_rel = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"
content_rel = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

print("Setup complete — ready to continue.")

Mounted at /content/drive
Paste your Hugging Face read token: ··········
Loaded pairs
Loaded client_cutoffs
Loaded content_daily


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded content_daily_position
Loaded pair_content_ids


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded content_queries


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded filtered_cq
Loaded query_popularity
Loaded query_set_sizes
Setup complete — ready to continue.


In [8]:
import os
save_dir = '/content/drive/MyDrive/flyrank_capstone'
print(os.listdir(save_dir))

['pairs.parquet', 'client_cutoffs.parquet', 'content_daily.parquet', 'content_daily_position.parquet', 'pair_content_ids.parquet', 'content_queries.parquet', 'filtered_cq.parquet', 'query_popularity.parquet', 'query_set_sizes.parquet', 'share_trend.parquet', 'position_corr.parquet', 'freshness_gap.parquet', 'ctr_gap.parquet', 'labels.parquet']


In [ ]:
con.execute("DROP SECRET IF EXISTS __default_huggingface")

from getpass import getpass
hf_token = getpass("Paste your Hugging Face read token: ")
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

Paste your Hugging Face read token: ··········


In [ ]:

#get position tier adjusted expected CTR.
con.execute("""
    CREATE OR REPLACE TABLE position_ctr_baseline AS
    SELECT
        ROUND(cdp.gsc_avg_position) AS position_tier,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS expected_ctr
    FROM content_daily_position cdp
    JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet') f
        ON cdp.client_hash_id = f.client_hash_id
        AND cdp.content_hash_id = f.content_hash_id
        AND cdp.report_date = f.report_date
    WHERE f.gsc_impressions > 0
    GROUP BY ROUND(cdp.gsc_avg_position)
""")
print(con.sql("SELECT * FROM position_ctr_baseline ORDER BY position_tier").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     position_tier  expected_ctr
0              0.0      0.001248
1              1.0      0.003353
2              2.0      0.005246
3              3.0      0.005953
4              4.0      0.005482
..             ...           ...
301          604.0      0.000000
302          606.0      0.000000
303          608.0      0.000000
304          807.0      0.000000
305          902.0      0.000000

[306 rows x 2 columns]


In [ ]:
ctr_gap = con.sql("""
WITH content_ctr AS (
    SELECT
        cdp.client_hash_id, cdp.content_hash_id,
        ROUND(AVG(cdp.gsc_avg_position)) AS avg_position_tier,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS actual_ctr
    FROM content_daily_position cdp
    JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet') f
        ON cdp.client_hash_id = f.client_hash_id
        AND cdp.content_hash_id = f.content_hash_id
        AND cdp.report_date = f.report_date
    WHERE f.gsc_impressions > 0
    GROUP BY cdp.client_hash_id, cdp.content_hash_id
),
content_with_gap AS (
    SELECT cc.client_hash_id, cc.content_hash_id, cc.actual_ctr,
           cc.actual_ctr - b.expected_ctr AS ctr_gap
    FROM content_ctr cc
    JOIN position_ctr_baseline b ON cc.avg_position_tier = b.position_tier
)
SELECT p.client_hash_id, p.content_a, p.content_b,
       ga.ctr_gap AS ctr_gap_a, gb.ctr_gap AS ctr_gap_b,
       ga.ctr_gap - gb.ctr_gap AS ctr_gap_difference
FROM pairs p
JOIN content_with_gap ga ON p.client_hash_id = ga.client_hash_id AND p.content_a = ga.content_hash_id
JOIN content_with_gap gb ON p.client_hash_id = gb.client_hash_id AND p.content_b = gb.content_hash_id
""").df()

print(f"Pairs with CTR-gap feature: {len(ctr_gap):,}")
ctr_gap.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pairs with CTR-gap feature: 50,469


,client_hash_id,content_a,content_b,ctr_gap_a,ctr_gap_b,ctr_gap_difference
0,client_62f4a7e64f5e0096,content_923f85eb9e3d092d,content_b6ee54e01f8a0795,-0.000537,-0.003714,0.003177
1,client_62f4a7e64f5e0096,content_58e5e8b95c2cb458,content_d6dc87267a497ca5,-0.002876,-0.001036,-0.001840
2,client_62f4a7e64f5e0096,content_854ee36f46e6ead7,content_f4acc6487599df18,0.000208,-0.001517,0.001724
3,client_62f4a7e64f5e0096,content_48efa39f9f518ed6,content_59c28998441fd713,-0.001818,-0.004294,0.002476
4,client_08a6a72ff48e62c0,content_77b8e423fd357440,content_bfad7f7df8620107,0.004307,-0.000703,0.005009


In [ ]:
#Saving the table to the drive

ctr_gap.to_parquet(f'{save_dir}/ctr_gap.parquet')

In [3]:
#Labels-postcutoff daily impressions(Flipped to >= instead of <)
con.execute("""
    CREATE OR REPLACE TABLE content_daily_post AS
    SELECT f.client_hash_id, f.content_hash_id, f.report_date, f.gsc_impressions
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet') f
    JOIN pair_content_ids pc
        ON f.client_hash_id = pc.client_hash_id AND f.content_hash_id = pc.content_hash_id
    JOIN client_cutoffs c ON f.client_hash_id = c.client_hash_id
    WHERE f.report_date >= c.cutoff_date AND f.gsc_data_available IS TRUE
""")
print(con.sql("SELECT COUNT(*) FROM content_daily_post").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   count_star()
0       7550710


In [5]:
#post-cutoff share_a per pair, then compare to pre-cutoff share_a to build the label
labels = con.sql("""
WITH post_pair_daily AS (
    SELECT
        p.client_hash_id, p.content_a, p.content_b, da.report_date,
        da.gsc_impressions AS impr_a, db.gsc_impressions AS impr_b
    FROM pairs p
    JOIN content_daily_post da ON p.client_hash_id = da.client_hash_id AND p.content_a = da.content_hash_id
    JOIN content_daily_post db ON p.client_hash_id = db.client_hash_id AND p.content_b = db.content_hash_id
        AND da.report_date = db.report_date
),
post_share AS (
    SELECT client_hash_id, content_a, content_b,
        AVG(CASE WHEN (impr_a + impr_b) > 0 THEN impr_a * 1.0 / (impr_a + impr_b) ELSE NULL END) AS post_avg_share_a,
        COUNT(*) AS n_days_post
    FROM post_pair_daily
    GROUP BY client_hash_id, content_a, content_b
    HAVING COUNT(*) >= 10
)
SELECT
    ps.client_hash_id, ps.content_a, ps.content_b,
    st.avg_share_a AS pre_avg_share_a,
    ps.post_avg_share_a,
    ps.post_avg_share_a - st.avg_share_a AS share_shift,
    CASE WHEN ABS(ps.post_avg_share_a - st.avg_share_a) >= 0.15 THEN 1 ELSE 0 END AS is_cannibalization_label
FROM post_share ps
JOIN share_trend st ON ps.client_hash_id = st.client_hash_id
    AND ps.content_a = st.content_a AND ps.content_b = st.content_b
""").df()

print(f"Total labeled pairs: {len(labels):,}")
print(f"Positive labels: {labels['is_cannibalization_label'].sum():,} ({labels['is_cannibalization_label'].mean()*100:.1f}%)")
labels.head()

,client_hash_id,content_a,content_b,pre_avg_share_a,post_avg_share_a,share_shift,is_cannibalization_label
0,client_a80fca3f171ed1de,content_25c839c8c5d19c79,content_3f3009be143a6b6e,0.857761,0.827781,-0.029979,0
1,client_a80fca3f171ed1de,content_b1b59f8df5bde059,content_c014d584363dfbf8,0.490947,0.673804,0.182857,1
2,client_9958f0a7ae1df715,content_1533331dca1461ed,content_cb0356a93e45c2bd,0.503022,0.662355,0.159332,1
3,client_a80fca3f171ed1de,content_5a41b683627ad220,content_c2d2b457b2425f87,0.650582,0.633994,-0.016587,0
4,client_a80fca3f171ed1de,content_4764f025e3562cdf,content_c0615378ad09d154,0.698100,0.717723,0.019622,0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, f1_score

feature_cols = ['jaccard', 'share_a_trend_slope', 'avg_share_a', 'position_correlation',
                 'freshness_gap_days', 'age_gap_days', 'ctr_gap_difference']
X = training_table[feature_cols]
y = training_table['is_cannibalization_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- Model ---
model = lgb.LGBMClassifier(random_state=42)
model.fit(X_train, y_train)
model_preds = model.predict(X_test)
model_probs = model.predict_proba(X_test)[:, 1]

print("=== LightGBM ===")
print(classification_report(y_test, model_preds))
print(f"ROC-AUC: {roc_auc_score(y_test, model_probs):.3f}")

# --- Baseline: simple rule, no model at all ---
# Rule: high query overlap + negative position correlation = flag as cannibalization
baseline_preds = ((X_test['jaccard'] > 0.5) & (X_test['position_correlation'] < -0.3)).astype(int)

print("\n=== Baseline Rule ===")
print(classification_report(y_test, baseline_preds))

# --- Direct comparison ---
print("\n=== Comparison ===")
print(f"Model F1:    {f1_score(y_test, model_preds):.3f}")
print(f"Baseline F1: {f1_score(y_test, baseline_preds):.3f}")

NameError: name 'pairs' is not defined

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.